# VidFetch Telegram bot

**Run both cells, enter your bot token, then send your bot a video link in Telegram.**

1. Use a standard **CPU** runtime (no GPU required).
2. Select **Runtime → Run all**. Setup can take a few minutes.
3. Enter the token supplied by [@BotFather](https://t.me/BotFather) in the hidden prompt.
4. Open the bot link printed below, send `/start`, then a video URL.
5. Choose a quality: buttons include estimated video + audio size. Large files are
   automatically compressed below the Telegram upload limit.
   Choose **Audio only** to receive just the audio as an M4A file in Telegram's audio player.

Keep the second cell running. Its stop button shuts the bot down; rerun it to restart.
Colab sessions are temporary and subject to [Colab's limits and policies](https://research.google.com/colaboratory/faq.html).
Interrupted downloads and old quality menus do not survive a runtime restart.
To update an older copy, stop its bot cell, reopen the latest GitHub notebook,
and run both cells again. Rerunning only the start cell does not update the embedded code.

Optional: restrict access with Telegram user IDs in cell 2 (use `/id` to find yours).
Leave the field blank to accept private messages from anyone who finds your bot.
Use `/cancel` to stop your request and `/status` to check it.

Sizes are estimates. Compression can reduce resolution, frame rate, and detail.
Some sites block Colab IP addresses or require sign-in. This notebook does not import cookies.
Download only content you are permitted to download.

[Source and documentation](https://github.com/HannanSeyfi/vidfetch-app/tree/codex/telegram-colab-bot/bot)


In [ ]:
#@title 1. Install tools and load VidFetch
import importlib
import os
from pathlib import Path
import shutil
import subprocess
import sys
import sysconfig
import tempfile

if not shutil.which("ffmpeg") or not shutil.which("ffprobe"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "--pre"]
               + ['httpx==0.28.1', 'yt-dlp[default,curl-cffi]>=2025.11.12', 'deno>=2.3,<3'], check=True)
os.environ["PATH"] = sysconfig.get_path("scripts") + os.pathsep + os.environ["PATH"]
encoders = subprocess.check_output(["ffmpeg", "-hide_banner", "-encoders"], text=True)
if "libx264" not in encoders:
    raise RuntimeError("This FFmpeg installation has no H.264 encoder (libx264).")
import deno
subprocess.run([deno.find_deno_bin(), "--version"], check=True)
# The notebook contains the same package as bot/vidfetch_bot in GitHub.
# Embedded sources make Open in Colab work from a branch, fork, or downloaded copy.
vidfetch_source_dir = Path(tempfile.mkdtemp(prefix="vidfetch-code-"))
for name, source in {'vidfetch_bot/__init__.py': '"""VidFetch\'s standalone Telegram companion. Android remains independent."""\n', 'vidfetch_bot/__main__.py': '"""Optional local entry point; Colab uses the same asynchronous run_bot function."""\n\nimport asyncio\nimport os\nfrom getpass import getpass\n\nfrom .app import run_bot\n\n\ndef main() -> None:\n    token = os.environ.get("TELEGRAM_BOT_TOKEN") or getpass("Telegram bot token from @BotFather: ")\n    try:\n        users = {\n            int(value.strip())\n            for value in os.environ.get("VIDFETCH_ALLOWED_USERS", "").split(",")\n            if value.strip()\n        }\n    except ValueError:\n        raise SystemExit(\n            "VIDFETCH_ALLOWED_USERS must contain comma-separated Telegram user IDs."\n        ) from None\n    try:\n        asyncio.run(run_bot(token, allowed_users=users))\n    except KeyboardInterrupt:\n        pass\n\n\nif __name__ == "__main__":\n    main()\n', 'vidfetch_bot/app.py': '"""Private-chat bot with expiring, owner-bound quality menus and bounded jobs."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport logging\nimport math\nimport re\nimport secrets\nimport shutil\nimport tempfile\nimport time\nfrom collections.abc import Callable\nfrom dataclasses import dataclass\nfrom html import escape\nfrom pathlib import Path\n\nfrom . import media\nfrom .telegram import Telegram, TelegramError\n\nHELP = (\n    "Send a link and choose a video quality or Audio only. Each button shows an estimated file size; "\n    "video sizes include audio where available. Audio only sends an M4A file in Telegram\'s audio player. "\n    "Oversized videos and audio are automatically compressed for Telegram.\\n\\n"\n    "Sizes are estimates; compression may reduce resolution and detail. "\n    "Only one request is processed at a time in this Colab session.\\n\\n"\n    "/cancel — cancel your current request\\n/status — show your request status\\n"\n    "/id — show your Telegram user ID\\n/help — show these instructions"\n)\n\n\ndef delivery_caption(title: str, details: str, url: str) -> str:\n    """Keep the full source URL clickable, including URLs beyond the caption limit."""\n    visible = f"{title}\\n{details}\\n\\nSource: {url}"\n    # Telegram counts entities in UTF-16 units. Use a short link label when the\n    # full visible URL would exceed the caption budget; the href stays intact.\n    label = url if len(visible.encode("utf-16-le")) // 2 <= 1024 else "Original link"\n    return (\n        f"{escape(title)}\\n{escape(details)}\\n\\n"\n        f\'Source: <a href="{escape(url, quote=True)}">{escape(label)}</a>\'\n    )\n\n\n@dataclass\nclass Selection:\n    owner: tuple[int, int]\n    url: str\n    analysis: media.Analysis\n    expires: float\n\n\n@dataclass\nclass Job:\n    task: asyncio.Task\n    status: str = "Waiting in the queue…"\n\n\nclass Bot:\n    def __init__(\n        self,\n        api: Telegram,\n        *,\n        allowed_users: set[int] | None = None,\n        workspace: Path | None = None,\n        max_jobs: int = 5,\n    ):\n        self.api = api\n        self.allowed_users = allowed_users or set()\n        self.workspace = workspace\n        self.max_jobs = max_jobs\n        self.jobs: dict[tuple[int, int], Job] = {}\n        self.selections: dict[str, Selection] = {}\n        self.slot = asyncio.Semaphore(1)\n\n    def expire_menus(self, owner: tuple[int, int] | None = None) -> None:\n        now = time.monotonic()\n        self.selections = {\n            key: value\n            for key, value in self.selections.items()\n            if value.expires > now and value.owner != owner\n        }\n        # Bound memory even if many people discover an unrestricted bot.\n        while len(self.selections) >= 200:\n            self.selections.pop(next(iter(self.selections)))\n\n    async def handle(self, update: dict) -> None:\n        callback = update.get("callback_query")\n        message = callback.get("message", {}) if callback else update.get("message", {})\n        user = callback.get("from", {}) if callback else message.get("from", {})\n        chat = message.get("chat", {})\n        if not user.get("id") or not chat.get("id"):\n            return\n        if chat.get("type") != "private":\n            if callback:\n                await self.api.acknowledge(callback["id"], "Use the bot in a private chat.")\n            return\n        owner = (chat["id"], user["id"])\n        if self.allowed_users and user["id"] not in self.allowed_users:\n            if callback:\n                await self.api.acknowledge(callback["id"], "This bot is private.")\n            else:\n                await self.api.message(chat["id"], "This bot is private.")\n            return\n        if callback:\n            await self.choose(callback, owner)\n            return\n        text = (message.get("text") or "").strip()\n        command = text.split(maxsplit=1)[0].split("@")[0].lower() if text else ""\n        if command in {"/start", "/help"}:\n            await self.api.message(chat["id"], HELP)\n        elif command == "/id":\n            await self.api.message(chat["id"], f"Your Telegram user ID: {user[\'id\']}")\n        elif command == "/cancel":\n            self.expire_menus(owner)\n            job = self.jobs.get(owner)\n            if job:\n                job.task.cancel()\n                await asyncio.gather(job.task, return_exceptions=True)\n            await self.api.message(\n                chat["id"], "Cancelled." if job else "No active request. Old quality menus cleared."\n            )\n        elif command == "/status":\n            await self.api.message(\n                chat["id"],\n                self.jobs[owner].status\n                if owner in self.jobs\n                else "Ready. Send a link or choose a quality from your latest menu.",\n            )\n        elif text.startswith(("https://", "http://")) and len(text.split()) == 1:\n            await self.schedule(\n                owner, lambda status, folder: self.inspect(text, owner, status, folder)\n            )\n        else:\n            await self.api.message(chat["id"], "Send one video link, or use /help.")\n\n    async def schedule(\n        self, owner: tuple[int, int], action: Callable, *, status_message: dict | None = None\n    ) -> bool:\n        if owner in self.jobs:\n            await self.api.message(\n                owner[0], "You already have a request. Use /cancel before starting another."\n            )\n            return False\n        if len(self.jobs) >= self.max_jobs:\n            await self.api.message(\n                owner[0], "The queue is full. Please try again after a video finishes."\n            )\n            return False\n        notice = status_message or await self.api.message(owner[0], "Waiting in the queue…")\n\n        async def status(text: str) -> None:\n            if job := self.jobs.get(owner):\n                job.status = text\n            await self.api.edit(owner[0], notice["message_id"], text)\n\n        async def work() -> None:\n            try:\n                await status("Waiting in the queue…")\n                async with self.slot:\n                    with tempfile.TemporaryDirectory(\n                        prefix="vidfetch-", dir=self.workspace\n                    ) as folder:\n                        await action(status, Path(folder))\n            except asyncio.CancelledError:\n                await status("Cancelled.")\n                raise\n            except (media.MediaError, TelegramError) as error:\n                await status(str(error))\n            except Exception as error:\n                logging.getLogger(__name__).warning("Request failed (%s)", type(error).__name__)\n                await status("Something went wrong. Send the link again or try another quality.")\n            finally:\n                self.jobs.pop(owner, None)\n\n        task = asyncio.create_task(work())\n        self.jobs[owner] = Job(task)\n\n        def finished(done: asyncio.Task) -> None:\n            if (job := self.jobs.get(owner)) and job.task is done:\n                self.jobs.pop(owner, None)\n            if not done.cancelled() and (error := done.exception()):\n                logging.getLogger(__name__).warning("Request stopped (%s)", type(error).__name__)\n\n        task.add_done_callback(finished)\n        return True\n\n    async def inspect(\n        self, url: str, owner: tuple[int, int], status: media.Status, folder: Path\n    ) -> None:\n        await status("Finding available video qualities…")\n        analysis = await media.analyze(url, folder)\n        self.expire_menus(owner)\n        key = secrets.token_urlsafe(8)\n        self.selections[key] = Selection(owner, url, analysis, time.monotonic() + 1800)\n        keyboard = [\n            [{"text": quality.label, "callback_data": f"q:{key}:{index}"}]\n            for index, quality in enumerate(analysis.qualities)\n        ]\n        try:\n            await self.api.message(\n                owner[0],\n                f"{analysis.title}\\n\\nChoose a quality. Sizes include audio "\n                "when known. Audio only sends an M4A audio track. "\n                "“compress” means the estimate exceeds the upload budget; "\n                "the delivered file may have lower quality.\\n\\nMenu expires in 30 minutes.",\n                keyboard=keyboard,\n            )\n        except BaseException:\n            self.selections.pop(key, None)\n            raise\n        await status("Qualities ready. Choose a button in the message below.")\n\n    async def choose(self, callback: dict, owner: tuple[int, int]) -> None:\n        self.expire_menus()\n        match = re.fullmatch(r"q:([A-Za-z0-9_-]+):(\\d{1,3})", callback.get("data", ""))\n        selection = self.selections.get(match[1]) if match else None\n        index = int(match[2]) if match else -1\n        if (\n            not selection\n            or selection.owner != owner\n            or not 0 <= index < len(selection.analysis.qualities)\n        ):\n            await self.api.acknowledge(callback["id"], "This menu expired. Send the link again.")\n            return\n        await self.api.acknowledge(callback["id"])\n        quality = selection.analysis.qualities[index]\n\n        async def deliver(status: media.Status, folder: Path) -> None:\n            choice = (\n                "audio"\n                if quality.audio_only\n                else (f"{quality.height}p" if quality.height else "best available video")\n            )\n            await status(f"Downloading {choice}…")\n            source = await media.download(selection.url, quality, folder)\n            if quality.audio_only:\n                audio = await media.prepare_audio(source, status)\n                size = audio.path.stat().st_size\n                if size > media.UPLOAD_LIMIT:\n                    raise media.MediaError(\n                        "The prepared audio exceeds the upload budget. Try a shorter clip."\n                    )\n                await status(f"Uploading {media.format_size(size)} of audio to Telegram…")\n                details = f"Audio only · {media.format_size(size)}"\n                if audio.compressed:\n                    details += " · compressed for Telegram"\n                await self.api.upload_audio(\n                    owner[0],\n                    audio.path,\n                    delivery_caption(selection.analysis.title, details, selection.url),\n                    title=selection.analysis.title,\n                    duration=math.ceil(audio.probe.duration),\n                )\n                await status("Audio sent. Send another link whenever you\'re ready.")\n                return\n            prepared = await media.prepare_video(source, status)\n            size = prepared.path.stat().st_size\n            if size > media.UPLOAD_LIMIT:\n                raise media.MediaError(\n                    "The prepared file exceeds the upload budget. Try a shorter video."\n                )\n            await status(f"Uploading {media.format_size(size)} to Telegram…")\n            result_quality = f"{prepared.probe.height}p"\n            details = f"{result_quality} · {media.format_size(size)}"\n            if prepared.compressed:\n                details += " · compressed for Telegram"\n            await self.api.upload(\n                owner[0],\n                prepared.path,\n                delivery_caption(selection.analysis.title, details, selection.url),\n                width=prepared.probe.width,\n                height=prepared.probe.height,\n                duration=math.ceil(prepared.probe.duration),\n            )\n            await status("Sent. Send another video link whenever you\'re ready.")\n\n        if await self.schedule(owner, deliver, status_message=callback["message"]):\n            self.selections.pop(match[1], None)\n\n    async def close(self) -> None:\n        tasks = [job.task for job in self.jobs.values()]\n        for task in tasks:\n            task.cancel()\n        await asyncio.gather(*tasks, return_exceptions=True)\n        self.jobs.clear()\n        self.selections.clear()\n\n\nasync def run_bot(token: str, *, allowed_users: set[int] | None = None) -> None:\n    """Await directly in Colab; interrupt the running cell to shut down cleanly."""\n    if not re.fullmatch(r"\\d{5,20}:[A-Za-z0-9_-]{20,}", token.strip()):\n        print("Enter the complete bot token from @BotFather, including the colon.")\n        return\n    # Even DEBUG logging must not expose URLs containing Telegram credentials.\n    for name in ("httpx", "httpcore"):\n        logging.getLogger(name).setLevel(logging.CRITICAL)\n    for tool in ("ffmpeg", "ffprobe"):\n        if not shutil.which(tool):\n            print(f"Missing {tool}. Run the setup cell first.")\n            return\n    api = Telegram(token.strip())\n    bot = Bot(api, allowed_users=allowed_users)\n    try:\n        identity = await api.call("getMe")\n        await api.call("deleteWebhook", {"drop_pending_updates": False})\n        await api.call(\n            "setMyCommands",\n            {\n                "commands": [\n                    {"command": "start", "description": "Send a link and choose video or audio"},\n                    {"command": "cancel", "description": "Cancel your current request"},\n                    {"command": "status", "description": "Show request status"},\n                    {"command": "id", "description": "Show your Telegram user ID"},\n                    {"command": "help", "description": "Show instructions"},\n                ]\n            },\n        )\n        print(\n            f"VidFetch is running: https://t.me/{identity[\'username\']}\\n"\n            "Open that bot in Telegram and send /start, then a video link.\\n"\n            "Keep this cell running. Use its stop button to stop the bot."\n        )\n        async for update in api.poll():\n            try:\n                await bot.handle(update)\n            except TelegramError as error:\n                if error.code in {401, 409}:\n                    raise\n    except TelegramError as error:\n        print(str(error))\n    except (asyncio.CancelledError, KeyboardInterrupt):\n        pass\n    finally:\n        await bot.close()\n        await api.close()\n        print("VidFetch stopped. Run this cell again to restart it.")\n', 'vidfetch_bot/media.py': '"""Extraction, size estimates, and verified Telegram-compatible compression."""\n\nfrom __future__ import annotations\n\nimport asyncio\nimport ipaddress\nimport json\nimport math\nimport os\nimport re\nimport shutil\nimport signal\nimport socket\nimport sys\nfrom collections.abc import Awaitable, Callable\nfrom dataclasses import dataclass\nfrom pathlib import Path\nfrom urllib.parse import urlsplit\n\nUPLOAD_LIMIT = 49_000_000  # Below the cloud Bot API\'s 50 MB limit, in bytes.\nTARGET_BYTES = 46_000_000\nDOWNLOAD_LIMIT = 2_000_000_000\nMAX_DURATION = 4 * 60 * 60\nAUDIO_BITRATE = 192_000\nStatus = Callable[[str], Awaitable[None]]\n\n\nclass MediaError(Exception):\n    """A safe, actionable message that can be displayed in Telegram."""\n\n\n@dataclass(frozen=True)\nclass Quality:\n    height: int | None\n    selector: str\n    size: int | None\n    audio_only: bool = False\n\n    @property\n    def label(self) -> str:\n        resolution = (\n            "Audio only"\n            if self.audio_only\n            else (f"{self.height}p" if self.height else "Best available")\n        )\n        size = f"~{format_size(self.size)}" if self.size else "size unknown"\n        compress = " · compress" if self.size and self.size > UPLOAD_LIMIT else ""\n        return f"{resolution} · {size}{compress}"\n\n\n@dataclass(frozen=True)\nclass Analysis:\n    title: str\n    qualities: tuple[Quality, ...]\n\n\n@dataclass(frozen=True)\nclass Probe:\n    duration: float\n    width: int\n    height: int\n    video_codec: str\n    pixel_format: str\n    audio_codec: str | None\n    fps: float = 30\n    video_index: int = 0\n    audio_index: int | None = None\n\n\n@dataclass(frozen=True)\nclass PreparedVideo:\n    path: Path\n    probe: Probe\n    compressed: bool\n\n\n@dataclass(frozen=True)\nclass AudioProbe:\n    duration: float\n    audio_codec: str\n    bitrate: int\n    audio_index: int\n\n\n@dataclass(frozen=True)\nclass PreparedAudio:\n    path: Path\n    probe: AudioProbe\n    compressed: bool\n\n\ndef number(value: object) -> float:\n    try:\n        result = float(value)\n        return result if math.isfinite(result) and result > 0 else 0\n    except (TypeError, ValueError):\n        return 0\n\n\ndef format_size(size: int) -> str:\n    # Decimal MB matches the upload budget and avoids MiB/MB ambiguity.\n    return f"{size / 1_000_000:.1f} MB"\n\n\ndef estimate_size(fmt: dict, duration: float) -> int | None:\n    for key in ("filesize", "filesize_approx"):\n        if value := number(fmt.get(key)):\n            return math.ceil(value)\n    bitrate = number(fmt.get("tbr"))\n    if not bitrate:\n        bitrate = number(fmt.get("vbr")) + number(fmt.get("abr"))\n    return math.ceil(duration * bitrate * 1000 / 8) if duration and bitrate else None\n\n\ndef is_aac(codec: str | None) -> bool:\n    return codec == "aac" or bool(codec and codec.startswith("mp4a"))\n\n\ndef audio_bitrate(source_bitrate: float) -> int:\n    return min(AUDIO_BITRATE, max(32_000, int(source_bitrate))) if source_bitrate else AUDIO_BITRATE\n\n\ndef estimate_audio_size(fmt: dict, duration: float) -> int | None:\n    """Estimate the delivered audio, never the full video used as a fallback source."""\n    separate = fmt.get("vcodec") == "none"\n    if is_aac(fmt.get("acodec")) and separate:\n        return estimate_size(fmt, duration)\n    bitrate = number(fmt.get("abr")) or (number(fmt.get("tbr")) if separate else 0)\n    if not duration:\n        return None\n    if is_aac(fmt.get("acodec")):\n        return math.ceil(duration * bitrate * 1000 / 8 * 1.02) if bitrate else None\n    # Non-AAC sources use one known conversion bitrate, so the menu does not\n    # depend on source bitrate fields that ffprobe may omit (notably for Opus).\n    return math.ceil(duration * AUDIO_BITRATE / 8 * 1.02)\n\n\ndef qualities_from_info(info: dict) -> tuple[Quality, ...]:\n    """Ask yt-dlp to select formats so displayed sizes match the download selector."""\n    from yt_dlp import YoutubeDL\n\n    formats = [f for f in info.get("formats", [info]) if not f.get("has_drm")]\n    heights = sorted(\n        {int(h) for f in formats if f.get("vcodec") != "none" and (h := number(f.get("height")))},\n        reverse=True,\n    )\n    duration = number(info.get("duration"))\n    result = []\n    host = (urlsplit(info.get("webpage_url") or "").hostname or "").lower()\n    direct_mp4_first = host == "pornhub.com" or host.endswith(".pornhub.com")\n    # The extractor returns formats sorted by yt-dlp\'s configured preference.\n    context = {\n        "formats": formats,\n        "has_merged_format": any(\n            f.get("vcodec") != "none" and f.get("acodec") != "none" for f in formats\n        ),\n        "incomplete_formats": not any(f.get("vcodec") == "none" for f in formats),\n    }\n    with YoutubeDL({"quiet": True, "no_warnings": True}) as ydl:\n        for height in heights or [None]:\n            ceiling = f"[height<={height}]" if height else ""\n            selector = f"bv{ceiling}+ba/b{ceiling}/bv{ceiling}"\n            if direct_mp4_first:\n                selector = f"b[ext=mp4]{ceiling}/" + selector\n            selected = list(ydl.build_format_selector(selector)(context))\n            if not selected:\n                continue\n            chosen = selected[-1]\n            actual_height = int(number(chosen.get("height"))) or None\n            if any(q.height == actual_height for q in result):\n                continue\n            streams = chosen.get("requested_formats") or [chosen]\n            sizes = [estimate_size(stream, duration) for stream in streams]\n            size = sum(sizes) if all(s is not None for s in sizes) else None\n            # Use the exact selected IDs on re-extraction, rather than guessing again.\n            identifiers = [str(stream.get("format_id", "")) for stream in streams]\n            if not all(re.fullmatch(r"[\\w.-]+", value, flags=re.ASCII) for value in identifiers):\n                continue\n            result.append(Quality(actual_height, "+".join(identifiers), size))\n        # Download a standalone audio stream if available. Otherwise take the\n        # smallest-preference combined format, avoiding a full-resolution video download.\n        audio_formats = list(ydl.build_format_selector("ba")(context))\n        audio = (\n            audio_formats[-1]\n            if audio_formats\n            else next(\n                (f for f in formats if f.get("acodec") != "none" and f.get("vcodec") != "none"),\n                None,\n            )\n        )\n        if audio and audio.get("acodec") != "none":\n            identifier = str(audio.get("format_id", ""))\n            if re.fullmatch(r"[\\w.-]+", identifier, flags=re.ASCII):\n                result.append(\n                    Quality(None, identifier, estimate_audio_size(audio, duration), audio_only=True)\n                )\n    if not result:\n        raise MediaError("No downloadable video or audio was found for this link.")\n    return tuple(result)\n\n\nasync def validate_url(url: str) -> str:\n    try:\n        parts = urlsplit(url)\n        host = parts.hostname\n        port = parts.port\n        if parts.scheme not in {"http", "https"} or not host or parts.username or parts.password:\n            raise ValueError\n        if port not in {None, 80, 443} or len(url) > 4096:\n            raise ValueError\n        addresses = await asyncio.get_running_loop().getaddrinfo(\n            host, port or (443 if parts.scheme == "https" else 80), type=socket.SOCK_STREAM\n        )\n        if not addresses or any(\n            not ipaddress.ip_address(item[4][0]).is_global for item in addresses\n        ):\n            raise ValueError\n    except (ValueError, OSError):\n        raise MediaError("Send a public http:// or https:// video link.") from None\n    return url\n\n\nasync def command(\n    args: list[str], *, timeout: int, folder: Path, download_guard: bool = False\n) -> str:\n    """Cancel/timeout always reaps the child before the caller deletes its files."""\n    log_path = folder / "process.stderr"\n    with log_path.open("wb") as log:\n        process = await asyncio.create_subprocess_exec(\n            *args, stdout=asyncio.subprocess.PIPE, stderr=log, start_new_session=os.name != "nt"\n        )\n        communicate = asyncio.create_task(process.communicate())\n        try:\n            async with asyncio.timeout(timeout):\n                while not communicate.done():\n                    await asyncio.wait({communicate}, timeout=1)\n                    if download_guard:\n                        total = 0\n                        for path in folder.rglob("*"):\n                            try:\n                                if path.is_file():\n                                    total += path.stat().st_size\n                            except FileNotFoundError:\n                                pass  # yt-dlp removes intermediates when merging finishes.\n                        if total > DOWNLOAD_LIMIT or shutil.disk_usage(folder).free < 256_000_000:\n                            raise MediaError(\n                                "This download exceeds the session\'s 2 GB storage budget."\n                            )\n                output, _ = await communicate\n        except TimeoutError:\n            raise MediaError(\n                "This operation took too long. Try a shorter video or lower quality."\n            ) from None\n        finally:\n            if process.returncode is None:\n                if os.name == "nt":\n                    # yt-dlp can itself spawn FFmpeg; reap that child as well.\n                    killer = await asyncio.create_subprocess_exec(\n                        "taskkill",\n                        "/PID",\n                        str(process.pid),\n                        "/T",\n                        "/F",\n                        stdout=asyncio.subprocess.DEVNULL,\n                        stderr=asyncio.subprocess.DEVNULL,\n                    )\n                    await killer.wait()\n                else:\n                    try:\n                        os.killpg(process.pid, signal.SIGTERM)\n                    except ProcessLookupError:\n                        pass\n                try:\n                    await asyncio.wait_for(process.wait(), timeout=5)\n                except TimeoutError:\n                    try:\n                        if os.name == "nt":\n                            process.kill()\n                        else:\n                            os.killpg(process.pid, signal.SIGKILL)\n                    except ProcessLookupError:\n                        pass\n                    await process.wait()\n            if os.name != "nt":\n                # The parent may exit before a Deno/FFmpeg descendant releases stdout.\n                try:\n                    os.killpg(process.pid, signal.SIGKILL)\n                except ProcessLookupError:\n                    pass\n            if not communicate.done():\n                communicate.cancel()\n            await asyncio.gather(communicate, return_exceptions=True)\n    if process.returncode:\n        # Never forward extractor URLs, cookies or arbitrary subprocess logs to chats.\n        detail = log_path.read_text(encoding="utf-8", errors="replace").lower()\n        if "sign in" in detail or "cookies" in detail or "login" in detail:\n            raise MediaError(\n                "This site requires sign-in or rejected this session. Try a public video."\n            )\n        if "403" in detail or "429" in detail or "not a bot" in detail:\n            raise MediaError(\n                "The site blocked this Colab connection. Try again later or use the Android app."\n            )\n        if "requested format" in detail:\n            raise MediaError(\n                "That quality is no longer available. Send the link again to refresh it."\n            )\n        raise MediaError(\n            "The media tool could not finish. The link may be unsupported or unavailable."\n        )\n    if len(output) > 16_000_000:\n        raise MediaError("The site returned too much metadata. Send a single-video link.")\n    return output.decode("utf-8", errors="replace").strip()\n\n\ndef extractor_args() -> list[str]:\n    import deno\n\n    return [\n        sys.executable,\n        "-m",\n        "yt_dlp",\n        "--ignore-config",\n        "--no-cache-dir",\n        "--no-playlist",\n        "--playlist-items",\n        "1",\n        "--socket-timeout",\n        "25",\n        "--retries",\n        "3",\n        "--fragment-retries",\n        "3",\n        "--abort-on-unavailable-fragments",\n        "--no-progress",\n        "--no-warnings",\n        "--js-runtimes",\n        f"deno:{deno.find_deno_bin()}",\n        "--format-sort",\n        "res,vcodec:h264,acodec:aac",\n    ]\n\n\nasync def analyze(url: str, folder: Path) -> Analysis:\n    await validate_url(url)\n    raw = await command(\n        extractor_args() + ["--dump-single-json", "--skip-download", "--", url],\n        timeout=180,\n        folder=folder,\n    )\n    try:\n        info = json.loads(raw)\n        if info.get("_type") in {"playlist", "multi_video"} or info.get("entries") is not None:\n            raise MediaError("Send a single-video link rather than a playlist.")\n        if info.get("is_live") or info.get("live_status") == "is_live":\n            raise MediaError("Live streams are not supported. Send a finished video.")\n        if number(info.get("duration")) > MAX_DURATION:\n            raise MediaError("Choose a video shorter than four hours for this Colab session.")\n        return Analysis(str(info.get("title") or "Video")[:200], qualities_from_info(info))\n    except (ValueError, TypeError, KeyError):\n        raise MediaError("The site did not return usable video information.") from None\n\n\nasync def download(url: str, quality: Quality, folder: Path) -> Path:\n    # Recheck the initial host when a button is used later in the session.\n    await validate_url(url)\n    raw = await command(\n        extractor_args()\n        + [\n            "--format",\n            quality.selector,\n            "--merge-output-format",\n            "mkv",\n            "--max-filesize",\n            str(DOWNLOAD_LIMIT),\n            "--output",\n            str(folder / "source.%(ext)s"),\n            "--print",\n            "after_move:%(filepath)j",\n            "--",\n            url,\n        ],\n        timeout=1800,\n        folder=folder,\n        download_guard=True,\n    )\n    try:\n        path = Path(json.loads(raw.splitlines()[-1])).resolve()\n        if path.parent != folder.resolve() or not path.is_file() or not path.stat().st_size:\n            raise ValueError\n        if path.stat().st_size > DOWNLOAD_LIMIT:\n            raise MediaError("This download exceeds the session\'s 2 GB storage budget.")\n        return path\n    except (ValueError, IndexError, TypeError):\n        kind = "audio" if quality.audio_only else "video"\n        raise MediaError(f"The extractor did not produce a completed {kind} file.") from None\n\n\nasync def probe_audio(path: Path, *, audio_only: bool = False) -> AudioProbe:\n    raw = await command(\n        ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)],\n        timeout=30,\n        folder=path.parent,\n    )\n    try:\n        info = json.loads(raw)\n        streams = info.get("streams", [])\n        audio = next(s for s in streams if s.get("codec_type") == "audio")\n        if audio_only and any(s.get("codec_type") == "video" for s in streams):\n            raise ValueError\n        duration = number(audio.get("duration")) or number(info.get("format", {}).get("duration"))\n        if not duration or not audio.get("codec_name"):\n            raise ValueError\n        if duration > MAX_DURATION:\n            raise MediaError("Choose audio shorter than four hours for this Colab session.")\n        return AudioProbe(\n            duration, audio["codec_name"], int(number(audio.get("bit_rate"))), int(audio["index"])\n        )\n    except (ValueError, KeyError, StopIteration, TypeError):\n        raise MediaError(\n            "This file does not contain a valid audio track. Try another link."\n        ) from None\n\n\ndef check_audio_duration(original: AudioProbe, final: AudioProbe) -> None:\n    if abs(final.duration - original.duration) > max(2, original.duration * 0.01):\n        raise MediaError("The prepared audio was incomplete. Please try a shorter clip.")\n\n\nasync def prepare_audio(\n    source: Path, status: Status, *, limit: int = UPLOAD_LIMIT, target: int = TARGET_BYTES\n) -> PreparedAudio:\n    info = await probe_audio(source)\n    output = source.parent / "telegram.m4a"\n    base = [\n        "ffmpeg",\n        "-hide_banner",\n        "-loglevel",\n        "error",\n        "-nostdin",\n        "-y",\n        "-i",\n        str(source),\n        "-map",\n        f"0:{info.audio_index}",\n        "-vn",\n        "-sn",\n        "-dn",\n    ]\n    await status("Extracting audio for Telegram…")\n    compressed = False\n    if info.audio_codec == "aac":\n        # Copying only the audio also avoids needless compression when the source\n        # video is large but its AAC track already fits Telegram\'s upload budget.\n        await command(\n            base + ["-c:a", "copy", "-movflags", "+faststart", str(output)],\n            timeout=300,\n            folder=source.parent,\n        )\n        final = await probe_audio(output, audio_only=True)\n        check_audio_duration(info, final)\n        if output.stat().st_size <= limit:\n            return PreparedAudio(output, final, False)\n        compressed = True\n        await status("Compressing audio to fit Telegram\'s upload limit…")\n    preferred_rate = audio_bitrate(info.bitrate) if info.audio_codec == "aac" else AUDIO_BITRATE\n    for attempt in range(3):\n        budget_rate = int(min(target, limit * 0.94) * 8 / info.duration * 0.95 * 0.8**attempt)\n        rate = min(preferred_rate, budget_rate)\n        if rate < 16_000:\n            raise MediaError(\n                "This audio is too long to fit a single Telegram upload. Try a shorter clip."\n            )\n        if rate < preferred_rate or attempt:\n            compressed = True\n            await status("Compressing audio to fit Telegram\'s upload limit…")\n        await command(\n            base\n            + [\n                "-c:a",\n                "aac",\n                "-b:a",\n                str(rate),\n                "-ac",\n                "1" if rate < 64_000 else "2",\n                "-movflags",\n                "+faststart",\n                str(output),\n            ],\n            timeout=1800,\n            folder=source.parent,\n        )\n        final = await probe_audio(output, audio_only=True)\n        check_audio_duration(info, final)\n        if output.stat().st_size <= limit:\n            return PreparedAudio(output, final, compressed)\n    raise MediaError("This audio could not fit Telegram\'s limit. Try a shorter clip.")\n\n\nasync def probe(path: Path) -> Probe:\n    raw = await command(\n        ["ffprobe", "-v", "error", "-show_streams", "-show_format", "-of", "json", str(path)],\n        timeout=30,\n        folder=path.parent,\n    )\n    try:\n        info = json.loads(raw)\n        streams = info.get("streams", [])\n        video = next(\n            s\n            for s in streams\n            if s.get("codec_type") == "video" and not s.get("disposition", {}).get("attached_pic")\n        )\n        audio = next((s for s in streams if s.get("codec_type") == "audio"), None)\n        duration = number(info.get("format", {}).get("duration")) or number(video.get("duration"))\n        width, height = int(video["width"]), int(video["height"])\n        rotation = next(\n            (s.get("rotation", 0) for s in video.get("side_data_list", []) if "rotation" in s), 0\n        )\n        if abs(float(rotation)) % 180 == 90:\n            width, height = height, width\n        fps_parts = video.get("avg_frame_rate", "30/1").split("/")\n        fps = number(fps_parts[0]) / (number(fps_parts[-1]) or 1)\n        if not duration or width < 2 or height < 2:\n            raise ValueError\n        if duration > MAX_DURATION:\n            raise MediaError("Choose a video shorter than four hours for this Colab session.")\n        return Probe(\n            duration,\n            width,\n            height,\n            video.get("codec_name", ""),\n            video.get("pix_fmt", ""),\n            audio.get("codec_name", "") if audio else None,\n            fps or 30,\n            int(video.get("index", 0)),\n            int(audio["index"]) if audio else None,\n        )\n    except (ValueError, KeyError, StopIteration, TypeError):\n        raise MediaError("The downloaded file does not contain a valid, finished video.") from None\n\n\ndef encoding_plan(info: Probe, target_bytes: int, attempt: int = 0) -> tuple[int, int, str]:\n    total_rate = int(target_bytes * 8 / info.duration * 0.94 * 0.82**attempt)\n    audio_rate = (\n        (96_000 if total_rate >= 400_000 else 48_000 if total_rate >= 120_000 else 16_000)\n        if info.audio_codec\n        else 0\n    )\n    video_rate = total_rate - audio_rate\n    if video_rate < 8_000:\n        raise MediaError(\n            "This video is too long to fit a single Telegram upload. Try a shorter clip."\n        )\n    fps = min(info.fps, 30 if video_rate >= 250_000 else 20 if video_rate >= 80_000 else 10)\n    scale = min(\n        1,\n        1920 / max(info.width, info.height),\n        math.sqrt(video_rate / (max(fps, 1) * 0.07 * info.width * info.height)),\n    )\n    width = max(2, int(info.width * scale) // 2 * 2)\n    height = max(2, int(info.height * scale) // 2 * 2)\n    return video_rate, audio_rate, f"scale={width}:{height},setsar=1,fps={fps:.3f}"\n\n\nasync def prepare_video(\n    source: Path, status: Status, *, limit: int = UPLOAD_LIMIT, target: int = TARGET_BYTES\n) -> PreparedVideo:\n    info = await probe(source)\n    output = source.parent / "telegram.mp4"\n    base = ["ffmpeg", "-hide_banner", "-loglevel", "error", "-nostdin", "-y", "-i", str(source)]\n    if (\n        source.stat().st_size <= limit\n        and info.video_codec == "h264"\n        and info.pixel_format == "yuv420p"\n    ):\n        await status("Preparing your video for Telegram…")\n        await command(\n            base\n            + [\n                "-map",\n                f"0:{info.video_index}",\n                "-map",\n                f"0:{info.audio_index}?" if info.audio_index is not None else "0:a:0?",\n                "-sn",\n                "-dn",\n                "-c:v",\n                "copy",\n                "-c:a",\n                "copy" if info.audio_codec in {None, "aac"} else "aac",\n                "-movflags",\n                "+faststart",\n                str(output),\n            ],\n            timeout=300,\n            folder=source.parent,\n        )\n        final = await probe(output)\n        if output.stat().st_size <= limit:\n            return PreparedVideo(output, final, False)\n    await status("Compressing for Telegram… Quality may be reduced to fit the upload limit.")\n    for attempt in range(3):\n        rate, audio_rate, filters = encoding_plan(info, min(target, int(limit * 0.94)), attempt)\n        video_args = [\n            "-map",\n            f"0:{info.video_index}",\n            "-vf",\n            filters,\n            "-c:v",\n            "libx264",\n            "-preset",\n            "fast",\n            "-pix_fmt",\n            "yuv420p",\n            "-b:v",\n            str(rate),\n            "-threads",\n            "2",\n            "-passlogfile",\n            str(source.parent / "encode-pass"),\n        ]\n        await command(\n            base + video_args + ["-pass", "1", "-an", "-sn", "-dn", "-f", "null", os.devnull],\n            timeout=1800,\n            folder=source.parent,\n        )\n        audio_args = (\n            [\n                "-map",\n                f"0:{info.audio_index}?",\n                "-c:a",\n                "aac",\n                "-b:a",\n                str(audio_rate),\n                "-ac",\n                "1" if audio_rate < 48_000 else "2",\n            ]\n            if info.audio_codec\n            else ["-an"]\n        )\n        await command(\n            base\n            + video_args\n            + ["-pass", "2"]\n            + audio_args\n            + ["-sn", "-dn", "-movflags", "+faststart", "-metadata:s:v:0", "rotate=0", str(output)],\n            timeout=1800,\n            folder=source.parent,\n        )\n        final = await probe(output)\n        if output.stat().st_size <= limit:\n            # A complete clip must survive compression, including any original audio.\n            if abs(final.duration - info.duration) > max(2, info.duration * 0.01):\n                raise MediaError("The compressed video was incomplete. Please try a shorter clip.")\n            if info.audio_codec and not final.audio_codec:\n                raise MediaError("Audio was missing after compression. Please try another quality.")\n            return PreparedVideo(output, final, True)\n    raise MediaError("This video could not fit Telegram\'s limit. Try a shorter clip.")\n', 'vidfetch_bot/telegram.py': '"""Small asynchronous Telegram Bot API client; credentials never enter log output."""\n\nfrom __future__ import annotations\n\nimport asyncio\nfrom pathlib import Path\n\nimport httpx\n\n\nclass TelegramError(Exception):\n    def __init__(self, code: int, description: str, retry_after: int = 0):\n        self.code = code\n        self.retry_after = retry_after\n        # Telegram errors normally contain no token, but don\'t retain raw responses.\n        messages = {\n            401: "The bot token is invalid. Get a token from @BotFather and run again.",\n            409: "Another copy of this bot is running. Stop it before starting this copy.",\n            403: "Telegram could not deliver a message to this chat.",\n            429: "Telegram asked the bot to slow down. Try again shortly.",\n        }\n        self.description = description.lower()  # Used only to classify a video-format error.\n        super().__init__(messages.get(code, "Telegram could not complete the request. Try again."))\n\n\nclass Telegram:\n    def __init__(self, token: str, *, transport: httpx.AsyncBaseTransport | None = None):\n        self._client = httpx.AsyncClient(\n            base_url=f"https://api.telegram.org/bot{token}/",\n            transport=transport,\n            timeout=httpx.Timeout(60, connect=20, write=600),\n            follow_redirects=False,\n        )\n\n    async def close(self) -> None:\n        await self._client.aclose()\n\n    async def call(self, method: str, data: dict | None = None, *, files: dict | None = None):\n        try:\n            if files:\n                response = await self._client.post(method, data=data, files=files)\n            else:\n                response = await self._client.post(method, json=data or {})\n            body = response.json()\n        except (httpx.HTTPError, ValueError):\n            # httpx\'s exception text includes the URL, which includes the bot token.\n            raise TelegramError(0, "network") from None\n        if not body.get("ok"):\n            raise TelegramError(\n                int(body.get("error_code", response.status_code)),\n                str(body.get("description", "")),\n                int(body.get("parameters", {}).get("retry_after", 0)),\n            )\n        return body.get("result")\n\n    async def message(self, chat_id: int, text: str, *, keyboard: list | None = None) -> dict:\n        payload = {"chat_id": chat_id, "text": text, "link_preview_options": {"is_disabled": True}}\n        if keyboard is not None:\n            payload["reply_markup"] = {"inline_keyboard": keyboard}\n        return await self.call("sendMessage", payload)\n\n    async def edit(\n        self, chat_id: int, message_id: int, text: str, *, keyboard: list | None = None\n    ) -> None:\n        try:\n            await self.call(\n                "editMessageText",\n                {\n                    "chat_id": chat_id,\n                    "message_id": message_id,\n                    "text": text,\n                    "reply_markup": {"inline_keyboard": keyboard or []},\n                    "link_preview_options": {"is_disabled": True},\n                },\n            )\n        except TelegramError as error:\n            if error.code in {401, 409}:\n                raise\n            # Deleted status messages, duplicate edits and temporary network failures\n            # must not destroy a successfully downloaded/compressed file.\n\n    async def acknowledge(self, query_id: str, text: str = "") -> None:\n        try:\n            await self.call("answerCallbackQuery", {"callback_query_id": query_id, "text": text})\n        except TelegramError:\n            pass  # An old callback may expire before Telegram receives the answer.\n\n    async def upload(\n        self, chat_id: int, path: Path, caption: str, *, width: int, height: int, duration: int\n    ) -> None:\n        payload = {\n            "chat_id": str(chat_id),\n            "caption": caption,\n            "parse_mode": "HTML",\n            "supports_streaming": "true",\n            "width": str(width),\n            "height": str(height),\n            "duration": str(duration),\n        }\n        with path.open("rb") as video:\n            try:\n                await self.call(\n                    "sendVideo", payload, files={"video": ("VidFetch.mp4", video, "video/mp4")}\n                )\n            except TelegramError as error:\n                # Only an explicit format rejection justifies resending as a document.\n                # Never retry ambiguous network/timeouts: the first upload may have arrived.\n                if error.code != 400 or not any(\n                    word in error.description\n                    for word in (\n                        "video_content_type",\n                        "video_content_invalid",\n                        "wrong type of the web page content",\n                    )\n                ):\n                    raise\n                video.seek(0)\n                await self.call(\n                    "sendDocument",\n                    {"chat_id": str(chat_id), "caption": caption, "parse_mode": "HTML"},\n                    files={"document": ("VidFetch.mp4", video, "video/mp4")},\n                )\n\n    async def updates(self, offset: int):\n        return await self.call(\n            "getUpdates",\n            {"offset": offset, "timeout": 30, "allowed_updates": ["message", "callback_query"]},\n        )\n\n    async def upload_audio(\n        self, chat_id: int, path: Path, caption: str, *, title: str, duration: int\n    ) -> None:\n        payload = {\n            "chat_id": str(chat_id),\n            "caption": caption,\n            "parse_mode": "HTML",\n            "title": title,\n            "duration": str(duration),\n        }\n        with path.open("rb") as audio:\n            await self.call(\n                "sendAudio", payload, files={"audio": ("VidFetch.m4a", audio, "audio/mp4")}\n            )\n\n    async def poll(self):\n        offset = 0\n        backoff = 1\n        while True:\n            try:\n                updates = await self.updates(offset)\n                backoff = 1\n                for update in updates:\n                    yield update\n                    offset = max(offset, update["update_id"] + 1)\n            except TelegramError as error:\n                if error.code in {401, 409}:\n                    raise\n                await asyncio.sleep(min(max(backoff, error.retry_after), 60))\n                backoff = min(backoff * 2, 30)\n'}.items():
    destination = vidfetch_source_dir / name
    destination.parent.mkdir(parents=True, exist_ok=True)
    destination.write_text(source, encoding="utf-8")
sys.path.insert(0, str(vidfetch_source_dir))
# Rerunning setup must load this notebook's version, not a cached older package.
for name in list(sys.modules):
    if name == "vidfetch_bot" or name.startswith("vidfetch_bot."):
        del sys.modules[name]
importlib.invalidate_caches()
print("Setup complete. Run the next cell and enter your bot token in the hidden prompt.")


In [ ]:
#@title 2. Start the Telegram bot
# Optional: comma-separated numeric Telegram user IDs. Blank allows any private-chat user.
allowed_user_ids = "" #@param {type:"string"}

from getpass import getpass
from vidfetch_bot.app import run_bot

async def start_vidfetch():
    try:
        allowed = {int(value.strip()) for value in allowed_user_ids.split(",") if value.strip()}
    except ValueError:
        print("Use numeric user IDs separated by commas, or leave the field blank.")
        return
    # Hidden input is never written into notebook source or cell outputs.
    await run_bot(getpass("Bot token from @BotFather: ").strip(), allowed_users=allowed)

await start_vidfetch()
